# Module 03 — Partitionnement et bilan de la journée

**Formation Big Data — ANSD / Data Innovation Lab**

Dernier notebook du jour. Trois choses :

1. le **partitionnement** sur disque, et comment choisir sa clé ;
2. le **rejeu du comparatif** du bloc précédent, cette fois en Parquet ;
3. le **mur de ce matin**, revisité.

## 1. Préparation

In [1]:
%load_ext autoreload 
%autoreload 2
import sys
from pathlib import Path
try:
    sys.path.append(str(Path(__file__).parent.parent.resolve()))
except NameError:
    sys.path.append(str(Path.cwd().parent.resolve()))
import gc
import time
import pyarrow.parquet as pq
from tools.outils_mesure import FICHIER, contexte_machine, mesurer
import pandas as pd


DOSSIER_DONNEES = FICHIER.parent
PARQUET = DOSSIER_DONNEES / "individus.parquet"
PARTITIONNE = DOSSIER_DONNEES / "individus_par_region"

machine = contexte_machine()

Cœurs logiques    : 12
Cœurs physiques   : 6
Mémoire totale    : 16.0 Go
Mémoire libre     : 5.7 Go
Fichier individus : 1036 Mo


## 2. Partitionner par région

Jusqu'ici, un seul fichier. Le **partitionnement** consiste à répartir les
données dans une arborescence de dossiers, selon les valeurs d'une colonne.

*Cellules fournies — regardez et exécutez.*

In [2]:
df = pd.read_parquet(PARQUET)
df["region"] = df["region"].str.strip().str.title()

depart = time.perf_counter()
df.to_parquet(PARTITIONNE, partition_cols=["region"],
              compression="snappy", index=False)
print(f"Écriture partitionnée : {time.perf_counter() - depart:.1f} s")

del df
gc.collect()

Écriture partitionnée : 4.0 s


0

In [3]:
# L'arborescence produite
for dossier in sorted(PARTITIONNE.iterdir())[:5]:
    fichiers = list(dossier.glob("*.parquet"))
    taille = sum(f.stat().st_size for f in fichiers) / 1024**2
    print(f"{dossier.name:<28} {len(fichiers)} fichier(s), {taille:6.1f} Mo")
print(f"\n{len(list(PARTITIONNE.iterdir()))} dossiers au total")

region=Dakar                 1 fichier(s),   37.0 Mo
region=Diourbel              1 fichier(s),   18.1 Mo
region=Fatick                1 fichier(s),    9.6 Mo
region=K%C3%A9dougou         1 fichier(s),    3.9 Mo
region=Kaffrine              1 fichier(s),    7.3 Mo

14 dossiers au total


Le nom du dossier porte l'information : `region=Dakar`. La colonne `region`
n'est donc **plus stockée dans les fichiers** — elle est déduite du chemin. Un
filtre sur la région devient une simple sélection de dossiers, sans lire une
seule ligne de données.

In [4]:
# Fourni : l'élagage de partitions, mesuré
print("Filtre sur une région :")
m_partitionne = mesurer("partitionné", lambda: pd.read_parquet(
    PARTITIONNE, columns=["age", "sexe"], filters=[("region", "==", "Dakar")]))
m_simple = mesurer("fichier unique", lambda: pd.read_parquet(
    PARQUET, columns=["age", "sexe", "region"],
    filters=[("region", "==", "Dakar")]))

Filtre sur une région :
  partitionné     0.589 s   pic    156 Mo
  fichier unique    0.212 s   pic    370 Mo


### Choisir une clé de partitionnement

C'est une décision d'architecture, et elle se prend une fois pour toutes.

In [5]:
# Fourni : que se passerait-il avec une clé à forte cardinalité ?
df = pd.read_parquet(PARQUET, columns=["region", "departement", "age", "sexe"])
df["region"] = df["region"].str.strip().str.title()
echantillon = df.head(300_000)

ESSAI = DOSSIER_DONNEES / "essai_partition_departement"
echantillon.to_parquet(ESSAI, partition_cols=["region", "departement"],
                       compression="snappy", index=False)

fichiers = list(ESSAI.rglob("*.parquet"))
taille_moyenne = sum(f.stat().st_size for f in fichiers) / len(fichiers) / 1024
print(f"{len(fichiers)} fichiers produits, {taille_moyenne:.0f} Ko en moyenne")
print("(et ceci sur 300 000 lignes seulement)")

del df, echantillon
gc.collect()

46 fichiers produits, 15 Ko en moyenne
(et ceci sur 300 000 lignes seulement)


0

**Question 1.** Pourquoi des milliers de petits fichiers dégradent-ils les
performances, alors même que chacun est rapide à lire ?

*Votre réponse :* …

> On appelle cela le **problème des petits fichiers**. Chaque fichier porte un
> en-tête, des métadonnées, et impose au moteur une ouverture séparée : au-delà
> de quelques centaines, le coût de gestion dépasse le gain de sélectivité.
>
> Règles usuelles : partitionner sur une colonne **fréquemment filtrée**, à
> **faible cardinalité** (quelques dizaines de valeurs au plus), en visant des
> partitions d'au moins 100 Mo. La région convient ; le département est déjà
> discutable ; l'identifiant de ménage serait une faute.

In [6]:
# Ménage
import shutil
shutil.rmtree(ESSAI, ignore_errors=True)

## 3. Le rejeu du comparatif

Les mêmes outils qu'au bloc précédent, la même question — effectif et âge moyen
par région — mais sur Parquet.

> ⏱️ **Si le temps manque**, passez `MESURER = False` : des mesures de référence
> seront utilisées et vous pourrez enchaîner sur l'interprétation.

In [7]:
MESURER = True

# Mesures de référence (poste de démonstration, 2 millions de lignes)
REFERENCE = pd.DataFrame({
    "outil":   ["pandas", "pandas", "polars", "polars", "duckdb", "duckdb"],
    "format":  ["CSV", "Parquet", "CSV", "Parquet", "CSV", "Parquet"],
    "secondes": [4.041, 0.119, 1.230, 0.313, 1.850, 0.024],
})

In [ ]:
# Fourni — mesurez, pour chacun des trois outils, le temps nécessaire à
# obtenir l'effectif et l'âge moyen par région, sur le fichier PARQUET.
#
# Chaque outil doit être libre d'optimiser : c'est la comparaison
# « à objectif identique » du bloc précédent.
#
#   pandas : pd.read_parquet(PARQUET, columns=[...]).groupby(...)
#   polars : pl.scan_parquet(PARQUET) ... .collect()
#   duckdb : SELECT region, COUNT(*), AVG(age) FROM 'chemin.parquet' GROUP BY 1

import duckdb
import polars as pl

con = duckdb.connect()
con.sql("SET enable_progress_bar = false")
CHEMIN = str(PARQUET).replace("\\", "/")

if MESURER:
    resultats = [
        {"outil": "pandas", "format": "Parquet",
         "secondes": mesurer("pandas", lambda: (pd.read_parquet(PARQUET, columns=["age", "region"])
                                                .groupby("region")
                                                .agg({"region" : "count", "age" : "mean"})
                                                ))["secondes"]},
        {"outil": "polars", "format": "Parquet",
         "secondes": mesurer("polars", lambda: (pl.scan_parquet(PARQUET)
                                                .group_by("region")
                                                .agg(
                                                    pl.len().alias("effectif"),
                                                    pl.col("age").mean().alias("age_moyen"),
                                                )
                                                .collect()
                                                ))["secondes"]},
        {"outil": "duckdb", "format": "Parquet",
         "secondes": mesurer("duckdb", lambda: (con
                                                .sql("SELECT region, COUNT(*), AVG(age) FROM '../00-data/individus.parquet' GROUP BY 1")
                                                ))["secondes"]},
    ]
    mesures_parquet = pd.DataFrame(resultats)
else:
    mesures_parquet = REFERENCE.query("format == 'Parquet'")

mesures_parquet

  pandas          0.800 s   pic    703 Mo
  polars          0.306 s   pic   1097 Mo
  duckdb          0.002 s   pic    728 Mo


,outil,format,secondes
0,pandas,Parquet,0.800
1,polars,Parquet,0.306
2,duckdb,Parquet,0.002


In [11]:
# Fourni : confrontation avec les mesures du bloc précédent, en CSV.
# Reprenez vos propres chiffres si vous les avez conservés.
comparatif = (REFERENCE.pivot(index="outil", columns="format", values="secondes")
                       .reindex(["pandas", "polars", "duckdb"]))
comparatif["gain"] = (comparatif["CSV"] / comparatif["Parquet"]).round(1)
comparatif

format,CSV,Parquet,gain
outil,,,
pandas,4.041,0.119,34.0
polars,1.230,0.313,3.9
duckdb,1.850,0.024,77.1


**Question 3.** Le passage à Parquet profite-t-il aux trois outils ? Dans
les mêmes proportions ?

*Votre réponse :* …

**Question 4.** Comparez l'ampleur de ce gain à celui obtenu en changeant de
moteur au bloc précédent. Lequel des deux leviers a le plus d'effet ?

*Votre réponse :* …

## 4. Bilan de la journée

Ce matin, vous avez rencontré **deux murs** :

- la **mémoire** — tout le fichier doit tenir en RAM, et il faut de la place en
  plus pour le lire, le trier, le joindre ;
- le **mono-cœur** — un seul processeur travaille, quelle que soit la machine.

Vous avez ensuite exploré **trois réponses** :

| Levier | Ce qu'il apporte | Ce qu'il coûte |
|---|---|---|
| Changer de moteur (Polars, DuckDB) | Parallélisme, optimisation du plan | Apprendre une nouvelle syntaxe |
| Découper le travail (Dask) | Passage à plusieurs machines | Complexité, surcoût de coordination |
| Changer de format (Parquet) | Lire moins, typer, sauter des blocs | Fichiers non lisibles à l'œil |

Et vous avez mesuré, sur vos propres données, que le **format** est le levier
qui rapporte le plus — et le seul qui profite à tous les outils à la fois.

**Trois idées à emporter**

1. **Mesurer avant de choisir.** Vous savez maintenant le faire, et vous avez
   les outils pour le refaire sur vos propres fichiers.
2. **La façon d'écrire le calcul** pèse souvent plus lourd que l'outil retenu :
   laisser le moteur optimiser plutôt que tout charger.
3. **Le moteur se remplace, le format reste.** Vous changerez d'outil plusieurs
   fois ; les fichiers que vous produirez cette année seront encore là dans dix
   ans. C'est la décision la plus engageante que vous prendrez.

**À compléter — ma synthèse personnelle**

- L'outil que j'utiliserai par défaut désormais : …
- Le format dans lequel je stockerai mes données de travail : …
